## 1. Setup

In [184]:
import json
import re

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, precision_recall_curve
from sklearn.model_selection import train_test_split

from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

## 2. Loading data

In [185]:
# Loading the dataset. gtin is read as a string so the codes never round-trip
# through float64, which would append a trailing '.0' to every barcode.
df = pd.read_csv('../Data/dataset.csv', dtype={'gtin': 'string'})
df['gtin'] = df['gtin'].replace('NA', pd.NA)

print(f'{len(df):,} SKUs | {df.gtin.notna().sum():,} carry a GTIN '
      f'({df.gtin.notna().mean():.1%}) | {df.gtin.nunique():,} distinct GTINs')
df.head()

71,623 SKUs | 30,078 carry a GTIN (42.0%) | 14,997 distinct GTINs


,sku_id,retailer,country,sku_name_eng,description_short_eng,breadcrumbs_eng,sku_url,image_url,sku_last_price,gtin,brand,category,attribute
0,6916,Walmart,USA,"Concord Foods, Smoothie Banana Drink Mix, 2 oz",""""" \n* Smoothie, Banana \n* Just Add: 1 ripe b...","["""" Food"""","""" Beverages"""","""" Powdered Drink Mi...",https://www.walmart.com/ip/Concord-Foods-Smoot...,https://i5.walmartimages.com/seo/Concord-Foods...,5.77,<NA>,by Concord Foods,Powder Concentrates,Free From: no allergens; Weight: 57; Flavour: ...
1,12782,Rite Aid,USA,"Sierra Mist Soda, Lemon- Lime, 12 fl oz - 12 ct",With other natural flavors. No artificial ingr...,"["""" Shop"""","""" Household"""","""" Food, Snacks & Ca...",https://www.riteaid.com/shop/sierra-mist-soda-...,https://www.riteaid.com/shop/media/catalog/pro...,6.11,12000227165,Sierra Mist,Lemonade/Lime,"Water Type: soda; Flavour: lemon, lime; Volume..."
2,16627,peapod,USA,Teavana Passion Tango Craft Herbal Iced Tea,Teavana Craft Iced Herbal Tea Passion Tango . ...,"["""" Beverages"""","""" Ready To Drink Tea""""]",https://www.peapod.com/,https://i5.peapod.com/c/K6/K6RMM.jpg,2.49,18200258763,Teavana,Still RTD Tea,Sweetener: sugar; Pack Material Type: Glass; C...
3,19126,Ocado,United Kingdom,Brecon Carreg Still Water,Nestled in the beautiful Brecon Beacons Nation...,"["""" Soft Drinks, Tea & Coffee"""","""" Water""""]",https://www.ocado.com/webshop/product/Brecon-C...,https://www.ocado.com/productImages/450/450760...,0.65,<NA>,Brecon Carreg,Still Bottled Water,Juice Content: 0-2%; Pack Material Type: Plast...
4,27474,Publix,USA,Zephyrhills 100% Natural Spring Water,ZEPHYRHILLS Brand 100% Natural Spring Water ha...,"["""" Beverages"""","""" Water, Seltzer & Sparkling ...",https://delivery.publix.com/store/items/41529659,https://d2lnr5mha7bycj.cloudfront.net/product-...,1.53,<NA>,Zephyrhills,Carbonated Bottled Water,Volume: 3000; Water Type: spring; Naturally de...


## 3. Feature processing

### Splitting up attribute column into several features

In [186]:
# Function for parsing the attribute column into a dictionary of sets. The input is a string
def parse_attributes(s):
    """'A: x, y; B: z' -> {'A': frozenset({'x', 'y'}), 'B': frozenset({'z'})}"""
    parsed = {}
    if not isinstance(s, str):
        return parsed
    for part in s.split(';'):
        if ':' not in part:
            continue
        name, _, value = part.partition(':')
        values = frozenset(v.strip().lower() for v in value.split(',') if v.strip())
        if values:
            parsed[name.strip()] = values
    return parsed

# Mapping the attribute column to dictionaries of sets, one per SKU.
attribute_maps = df['attribute'].map(parse_attributes)

# Ordering attribute names by coverage so the best-populated columns come first.
attributes = (pd.Series([name for row in attribute_maps for name in row])
                .value_counts()
                .index
                .tolist())

# Creating one column per attribute, with the values being the set of values for that attribute.
for attribute in attributes:
    df[attribute] = attribute_maps.map(lambda row, a=attribute: row.get(a, pd.NA))

# 'Special Edition' and 'Giftbox' only ever take one value, so they carry nothing
# as strings - but their *presence* is a real signal (216 and 21 SKUs). Therefore
# we create boolean columns for them instead of string columns.
flags = ['Special Edition', 'Giftbox']
for flag in flags:
    df[flag] = df[flag].notna()
attributes = [a for a in attributes if a not in flags]

# Dropping columns. Only sku_url has no bearing on whether two SKUs are the same
# item. image_url and sku_last_price are kept as pair features: where two SKUs
# share an image_url exactly, 94.5% of them are true matches against a 4.1% base
# rate, and price ratio separates the classes by 0.17.
df = df.drop(columns=['attribute', 'sku_url'])

print(f'{len(attributes)} attributes + {len(flags)} presence flags')
df.head()

35 attributes + 2 presence flags


,sku_id,retailer,country,sku_name_eng,description_short_eng,breadcrumbs_eng,image_url,sku_last_price,gtin,brand,...,Weight,Botanicals And Functional Ingredients,Coffee Type,Sports Positioning,Count per Unit,Roast Type,Sports Drinks Style,Nutri-Score,Special Edition,Giftbox
0,6916,Walmart,USA,"Concord Foods, Smoothie Banana Drink Mix, 2 oz",""""" \n* Smoothie, Banana \n* Just Add: 1 ripe b...","["""" Food"""","""" Beverages"""","""" Powdered Drink Mi...",https://i5.walmartimages.com/seo/Concord-Foods...,5.77,<NA>,by Concord Foods,...,frozenset({57}),<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,False,False
1,12782,Rite Aid,USA,"Sierra Mist Soda, Lemon- Lime, 12 fl oz - 12 ct",With other natural flavors. No artificial ingr...,"["""" Shop"""","""" Household"""","""" Food, Snacks & Ca...",https://www.riteaid.com/shop/media/catalog/pro...,6.11,12000227165,Sierra Mist,...,<NA>,<NA>,<NA>,<NA>,frozenset({12}),<NA>,<NA>,<NA>,False,False
2,16627,peapod,USA,Teavana Passion Tango Craft Herbal Iced Tea,Teavana Craft Iced Herbal Tea Passion Tango . ...,"["""" Beverages"""","""" Ready To Drink Tea""""]",https://i5.peapod.com/c/K6/K6RMM.jpg,2.49,18200258763,Teavana,...,<NA>,frozenset({licorice}),<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,False,False
3,19126,Ocado,United Kingdom,Brecon Carreg Still Water,Nestled in the beautiful Brecon Beacons Nation...,"["""" Soft Drinks, Tea & Coffee"""","""" Water""""]",https://www.ocado.com/productImages/450/450760...,0.65,<NA>,Brecon Carreg,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,False,False
4,27474,Publix,USA,Zephyrhills 100% Natural Spring Water,ZEPHYRHILLS Brand 100% Natural Spring Water ha...,"["""" Beverages"""","""" Water, Seltzer & Sparkling ...",https://d2lnr5mha7bycj.cloudfront.net/product-...,1.53,<NA>,Zephyrhills,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,False,False


### Parsing the breadcrumbs_eng column

In [187]:
# Function for parsing the breadcrumbs column into a tuple of strings. The input is a string
def parse_breadcrumbs(s):
    """'["" A"","" B""]' -> ('A', 'B'). Tolerates NaN, bare strings, inner commas."""
    if not isinstance(s, str) or not s.strip():
        return ()
    text = s.strip()
    if text.startswith('['):
        try:
            parsed = json.loads(text.replace('""', '"'))
            return tuple(x.strip() for x in parsed if isinstance(x, str) and x.strip())
        except (json.JSONDecodeError, TypeError):
            pass
    return tuple(x.strip() for x in re.findall(r'[^",\[\]]+', text) if x.strip())

# Mapping the breadcrumbs column to tuples of strings, one per SKU.
df['breadcrumbs'] = df['breadcrumbs_eng'].map(parse_breadcrumbs)
df['bc_depth'] = df['breadcrumbs'].str.len()

# Creating one column per breadcrumb level, with the values being the string at that level.
for level in range(3):
    df[f'bc_{level}'] = df['breadcrumbs'].map(
        lambda path, i=level: path[i].lower() if len(path) > i else pd.NA)

# Creating a column of sets of tokens in the breadcrumbs, for fuzzy matching.
df['bc_tokens'] = df['breadcrumbs'].map(
    lambda path: frozenset(re.findall(r'[a-z0-9]+', ' '.join(path).lower())))

# Dropping columns
df = df.drop(columns=['breadcrumbs_eng'])

print(f'{df.bc_depth.gt(0).sum():,} SKUs with breadcrumbs, '
      f'{(df.bc_depth == 0).sum()} without | median depth {df.bc_depth.median():.0f}')
df[['sku_id', 'breadcrumbs', 'bc_0', 'bc_1', 'bc_depth']].head()

71,518 SKUs with breadcrumbs, 105 without | median depth 3


,sku_id,breadcrumbs,bc_0,bc_1,bc_depth
0,6916,"(Food, Beverages, Powdered Drink Mixes, All Po...",food,beverages,4
1,12782,"(Shop, Household, Food, Snacks & Candy, Food, ...",shop,household,5
2,16627,"(Beverages, Ready To Drink Tea)",beverages,ready to drink tea,2
3,19126,"(Soft Drinks, Tea & Coffee, Water)","soft drinks, tea & coffee",water,2
4,27474,"(Beverages, Water, Seltzer & Sparkling Water)",beverages,"water, seltzer & sparkling water",2


### Normalizing string columns

In [188]:
# Normalizing text columns for blocking and fuzzy matching. Lowercasing, punctuation to spaces, whitespace collapsed
def norm_text(s):
    if not isinstance(s, str):
        return pd.NA
    return ' '.join(re.sub(r'[^a-z0-9]+', ' ', s.lower()).split())

# Mapping the text columns to their normalized forms.
df['name_norm'] = df['sku_name_eng'].map(norm_text)
df['desc_norm'] = df['description_short_eng'].map(norm_text)
df['brand_norm'] = (df['brand'].str.lower()
                               .str.replace(r'^by\s+', '', regex=True)
                               .map(norm_text))

df[['name_norm', 'brand_norm']].head()

,name_norm,brand_norm
0,concord foods smoothie banana drink mix 2 oz,concord foods
1,sierra mist soda lemon lime 12 fl oz 12 ct,sierra mist
2,teavana passion tango craft herbal iced tea,teavana
3,brecon carreg still water,brecon carreg
4,zephyrhills 100 natural spring water,zephyrhills


### Checking null counts and unique values in each column

In [189]:
# Column-level summary: coverage and cardinality of every engineered feature.
summary = pd.DataFrame({
    'Unique Values': {c: df[c].nunique() for c in df.columns},
    'Missing Values': df.isnull().sum(),
    'Type': {c: str(df[c].dtype) for c in df.columns}
})
summary['Coverage'] = (1 - summary['Missing Values'] / len(df)).map('{:.1%}'.format)
summary.index.name = 'Column Name'

summary.sort_values('Missing Values').reset_index()

,Column Name,Unique Values,Missing Values,Type,Coverage
0,sku_id,71623,0,int64,100.0%
1,retailer,280,0,str,100.0%
2,country,19,0,str,100.0%
3,image_url,63580,0,str,100.0%
4,category,24,0,str,100.0%
5,brand,2265,0,str,100.0%
6,Special Edition,2,0,bool,100.0%
7,breadcrumbs,8749,0,object,100.0%
8,bc_depth,14,0,int64,100.0%
9,Giftbox,2,0,bool,100.0%


## 4. Candidate generation (blocking)

### Creating Dictionary

In [190]:
# Blocking list storage
blocking_list = {}

# Looping through the rows of the dataframe to populate the blocking list based on normalized brand names.
for position, row in enumerate(df.itertuples()):
    if not isinstance(row.brand_norm, str):
        continue
    brand = row.brand_norm.replace(' ', '')
    if not brand:
        continue
    if brand not in blocking_list:
        blocking_list[brand] = []
    blocking_list[brand].append(position)

# Computing shelf sizes, total shelved SKUs, and candidate pairs for comparison.
shelf_sizes = pd.Series({brand: len(rows) for brand, rows in blocking_list.items()})
shelved = int(shelf_sizes.sum())
candidate_pairs = int((shelf_sizes * (shelf_sizes - 1) // 2).sum())

print(f'{len(blocking_list):,} shelves holding {shelved:,} of {len(df):,} SKUs '
      f'({len(df) - shelved} unshelved)')
print(f'shelf size: max {shelf_sizes.max()} | median {shelf_sizes.median():.0f} | '
      f'{shelf_sizes.eq(1).sum()} singletons')
print(f'{candidate_pairs:,} pairs to check '
      f'({len(df) * (len(df) - 1) / 2 / candidate_pairs:,.0f}x fewer than all-vs-all)')

2,258 shelves holding 71,623 of 71,623 SKUs (0 unshelved)
shelf size: max 343 | median 17 | 185 singletons
3,008,719 pairs to check (852x fewer than all-vs-all)


### Creating a list of pairs

In [191]:
# Creating a numpy array of pairs of positions to compare, one row per pair.
pairs = np.empty((candidate_pairs, 2), dtype=np.int32)

# Filling the pairs array with all combinations of positions within each shelf.
written = 0
for rows in blocking_list.values():
    for i in range(len(rows)):
        for j in range(i + 1, len(rows)):
            pairs[written] = (rows[i], rows[j])
            written += 1

assert written == candidate_pairs, f'expected {candidate_pairs} pairs, wrote {written}'
print(f'{len(pairs):,} pairs, {pairs.nbytes / 1e6:.0f} MB')


3,008,719 pairs, 24 MB


### Labelling pairs

In [192]:
# Splitting the pairs array into labelled and unlabelled pairs.
# Labelled pairs have a GTIN on BOTH sides, so the answer is already known - these become the training data. 
# Unlabelled pairs are missing at least one GTIN and are what the classifier has to decide.

gtin = df['gtin'].to_numpy()
pair_gtins = gtin[pairs]

labeled_mask = pd.notna(pair_gtins).all(axis=1)
labeled_pairs = pairs[labeled_mask]
unlabeled_pairs = pairs[~labeled_mask]

# The training label: 1 when the two barcodes agree, 0 when they differ.
y = (pair_gtins[labeled_mask, 0] == pair_gtins[labeled_mask, 1]).astype(np.int8)

print(f'labelled   {len(labeled_pairs):>10,} pairs  ({y.mean():.2%} positive, '
      f'{y.sum():,} matches)')
print(f'unlabelled {len(unlabeled_pairs):>10,} pairs  -> classifier decides')


labelled      853,135 pairs  (4.09% positive, 34,921 matches)
unlabelled  2,155,584 pairs  -> classifier decides


## 5. Feature Engineering

In [193]:
def smallest_number(value):
    """Smallest numeric entry of a set-valued attribute, else NaN."""
    if not isinstance(value, frozenset):
        return np.nan
    numbers = [float(v) for v in value if re.fullmatch(r'\d+(\.\d+)?', v)]
    return min(numbers) if numbers else np.nan

def both_equal(values):
    """1.0 where the two SKUs share a value, NaN where either side is missing."""
    left, right = values[LEFT], values[RIGHT]
    known = pd.notna(left) & pd.notna(right)
    return np.where(known, (left == right).astype(np.float32), np.nan)


def smaller_over_larger(values):
    """min/max, so 1.0 means identical and the order of the pair does not matter."""
    left, right = values[LEFT], values[RIGHT]
    high = np.fmax(left, right)
    with np.errstate(invalid='ignore', divide='ignore'):
        return np.where(high > 0, np.fmin(left, right) / high, np.nan).astype(np.float32)


def jaccard(sets):
    """Overlap of two token sets: |intersection| / |union|."""
    out = np.empty(len(pairs), dtype=np.float32)
    for k in range(len(pairs)):
        left, right = sets[LEFT[k]], sets[RIGHT[k]]
        union = len(left | right)
        out[k] = len(left & right) / union if union else np.nan
    return out


def cosine_in_chunks(matrix, step=500_000):
    """Row-wise cosine of the two TF-IDF vectors, chunked to bound memory."""
    out = np.empty(len(pairs), dtype=np.float32)
    for start in range(0, len(pairs), step):
        stop = min(start + step, len(pairs))
        out[start:stop] = np.asarray(
            matrix[LEFT[start:stop]].multiply(matrix[RIGHT[start:stop]]).sum(axis=1)).ravel()
    return out

def set_jaccard(values):
    """Overlap of two set-valued attributes, NaN where either side is missing.

    Graded rather than binary: {lemon} vs {lemon, lime} scores 0.5, not 0. Use this
    for word-valued attributes (Flavour, Carbonization, Pack Material Type) - they
    hold sets of strings, so smallest_number would return NaN for every row and
    both_equal would demand the sets be *identical* rather than merely overlapping.
    """
    out = np.empty(len(pairs), dtype=np.float32)
    for k in range(len(pairs)):
        left, right = values[LEFT[k]], values[RIGHT[k]]
        if isinstance(left, frozenset) and isinstance(right, frozenset):
            out[k] = len(left & right) / len(left | right)
        else:
            out[k] = np.nan
    return out

def containment(sets):
    """|intersection| / |smaller set|.

    Unlike Jaccard this does not punish a name for carrying extra words:
    "peace tea caddy shack" vs "peace tea caddy shack can 23 fl oz" scores 1.0 here
    but only 0.57 by Jaccard, because Jaccard divides by the union. One retailer
    padding a name with size and packaging words was the single most common reason
    the model missed a true match, so this feature targets that case directly.
    """
    out = np.empty(len(pairs), dtype=np.float32)
    for k in range(len(pairs)):
        left, right = sets[LEFT[k]], sets[RIGHT[k]]
        smaller = min(len(left), len(right))
        out[k] = len(left & right) / smaller if smaller else np.nan
    return out

In [194]:
# Turning each pair of row positions into numbers describing how similar the two SKUs are. 
LEFT, RIGHT = pairs[:, 0], pairs[:, 1]

# Compute per-SKU values that are used in the pair-wise comparisons.
# Volume is the only numeric attribute - it is always a plain integer count of ml.
# Carbonization, Flavour and Pack Material Type hold sets of WORDS, so they are kept
# as sets and compared with set_jaccard rather than converted to numbers.
volume = df['Volume'].map(smallest_number).to_numpy(dtype=float)
carbonization = df['Carbonization'].to_numpy()
flavour = df['Flavour'].to_numpy()
pack_material_type = df['Pack Material Type'].to_numpy()
price = pd.to_numeric(df['sku_last_price'], errors='coerce').to_numpy(dtype=float)
category = df['category'].to_numpy()
retailer = df['retailer'].to_numpy()
country = df['country'].to_numpy()
image_url = df['image_url'].to_numpy()
name_tokens = [frozenset(s.split()) if isinstance(s, str) else frozenset()
               for s in df['name_norm']]
name_length = df['name_norm'].fillna('').str.len().to_numpy(dtype=float)
crumb_tokens = df['bc_tokens'].to_numpy()

# Size and packaging words are already captured by the volume features, so stripping
# them stops them diluting the name comparison.
NOISE_WORDS = frozenset('fl oz ml l lt liter litre cl g kg pack pk ct count can cans '
                        'bottle bottles case box carton x of the a and'.split())
core_tokens = [frozenset(w for w in tokens
                         if w not in NOISE_WORDS and not w.isdigit())
               for tokens in name_tokens]

# Making a list of dictionaries, one per SKU, where each dictionary maps attribute names to their set of values.
sku_attributes = [dict() for _ in range(len(df))]
for attribute in attributes:
    for position, value in enumerate(df[attribute].to_numpy()):
        if isinstance(value, frozenset):
            sku_attributes[position][attribute] = value

# Using character n-grams for comparing names and tolerating spelling noise
tfidf = normalize(
    TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 3), min_df=3)
    .fit_transform(df['name_norm'].fillna('')))

# Attributes present on both sides either overlap (agreement) or are disjoint
# (a conflict, e.g. Flavour cherry vs lime) - the strongest negative signal we have.
shared = np.zeros(len(pairs), dtype=np.int8)
agreed = np.zeros(len(pairs), dtype=np.int8)
for k in range(len(pairs)):
    left, right = sku_attributes[LEFT[k]], sku_attributes[RIGHT[k]]
    if len(right) < len(left):
        left, right = right, left
    for attribute, values in left.items():
        other = right.get(attribute)
        if other is not None:
            shared[k] += 1
            if values & other:
                agreed[k] += 1

features = pd.DataFrame({
    'name_jaccard':    jaccard(name_tokens),
    'name_cosine':     cosine_in_chunks(tfidf),
    'name_length_ratio': smaller_over_larger(name_length),
    'name_containment': containment(name_tokens),
    'name_containment_core': containment(core_tokens),
    'volume_equal':    both_equal(volume),
    'volume_ratio':    smaller_over_larger(volume),
    'flavour_jaccard': set_jaccard(flavour),
    'carbonization_jaccard': set_jaccard(carbonization),
    'pack_material_jaccard': set_jaccard(pack_material_type),
    'price_ratio':     smaller_over_larger(price),
    'category_equal':  both_equal(category),
    'retailer_equal':  both_equal(retailer),
    'country_equal':   both_equal(country),
    'image_equal':     both_equal(image_url),
    'breadcrumb_jaccard': jaccard(crumb_tokens),
    'attr_shared':     shared,
    'attr_agreed':     agreed,
    'attr_conflicts':  shared - agreed,
    'attr_agree_ratio': np.divide(agreed, np.maximum(shared, 1), dtype=np.float32),
})

print(f'{features.shape[0]:,} pairs x {features.shape[1]} features '
      f'({features.memory_usage(deep=True).sum() / 1e6:.0f} MB)')
features.head()

3,008,719 pairs x 20 features (214 MB)


,name_jaccard,name_cosine,name_length_ratio,name_containment,name_containment_core,volume_equal,volume_ratio,flavour_jaccard,carbonization_jaccard,pack_material_jaccard,price_ratio,category_equal,retailer_equal,country_equal,image_equal,breadcrumb_jaccard,attr_shared,attr_agreed,attr_conflicts,attr_agree_ratio
0,0.500000,0.926802,0.659091,1.00,1.00,NaN,NaN,1.0,NaN,NaN,0.188908,1.0,0.0,1.0,0.0,0.375000,3,3,0,1.000000
1,0.333333,0.760184,0.659091,0.75,0.75,NaN,NaN,0.0,NaN,NaN,0.188908,1.0,0.0,1.0,0.0,0.375000,3,2,1,0.666667
2,0.181818,0.429833,0.886364,0.40,0.40,NaN,NaN,0.0,NaN,NaN,0.171577,0.0,0.0,1.0,0.0,0.090909,3,1,2,0.333333
3,0.000000,0.000000,0.409091,0.00,0.00,NaN,1.0,0.0,NaN,NaN,0.216638,0.0,0.0,1.0,0.0,0.000000,3,1,2,0.333333
4,0.500000,0.926802,0.659091,1.00,1.00,NaN,NaN,1.0,NaN,NaN,0.171577,1.0,0.0,1.0,0.0,0.000000,3,3,0,1.000000


## 6. Classifying pair candidates

### Training classifier

In [195]:
# Creating a numpy feature matrix for the labelled pairs, to be used for training the classifier.
X = features[labeled_mask].to_numpy(dtype=np.float32)

# Splitting the labelled pairs into a training set and a validation set, stratified by the label to maintain class balance.
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y)

# Using a histogram-based gradient boosting classifier
model = HistGradientBoostingClassifier(
    max_iter=900, learning_rate=0.05, max_leaf_nodes=127,
    min_samples_leaf=20, l2_regularization=1.0, random_state=0)

# Fitting the model to the training data
model.fit(X_train, y_train)

# Validating model performance on the held-out data
val_scores = model.predict_proba(X_val)[:, 1]

# Scoring model performance
print(f'average precision: {average_precision_score(y_val, val_scores):.4f}')

# Precision > recall as mispredicting can lead to growing damages (false positives chain unrelated items into one)
precision, recall, thresholds = precision_recall_curve(y_val, val_scores)
precision, recall = precision[:-1], recall[:-1]

print()
print(f"{'target P':>9}{'threshold':>11}{'precision':>11}{'recall':>9}{'F1':>8}")
for target in [0.80, 0.90, 0.95, 0.99]:
    meets = precision >= target
    if not meets.any():
        print(f'{target:>9.2f}{"unreachable":>11}')
        continue
    i = int(np.argmax(np.where(meets, recall, -1.0)))
    p, r, t = precision[i], recall[i], thresholds[i]
    print(f'{target:>9.2f}{t:>11.3f}{p:>11.3f}{r:>9.3f}{2*p*r/(p+r):>8.3f}')

average precision: 0.8426

 target P  threshold  precision   recall      F1
     0.80      0.406      0.800    0.728   0.762
     0.90      0.658      0.900    0.578   0.704
     0.95      0.797      0.950    0.458   0.618
     0.99      0.949      0.990    0.269   0.423


### Deciding every pair

In [196]:
# Setting the threshold for deciding whether a pair is a match or not. 
MATCH_THRESHOLD = 0.80

# Scoring every pair with the model, including the labelled ones for better threshold selection
model_scores = model.predict_proba(features.to_numpy(dtype=np.float32))[:, 1]

# Certainty where we have it, the model's probability everywhere else.
pair_scores = model_scores.copy()
pair_scores[labeled_mask] = y                      # exactly 0.0 or 1.0

# Deciding every candidate pair: match or not
is_match = pair_scores >= MATCH_THRESHOLD

# Counting how many pairs were decided by GTIN and how many by the model, and printing the results.
decided_by_gtin = int(labeled_mask.sum())
decided_by_model = int((~labeled_mask).sum())
print(f'{len(pairs):,} pairs decided:')
print(f'   {decided_by_gtin:>10,} by GTIN   ({int(is_match[labeled_mask].sum()):,} matches)')
print(f'   {decided_by_model:>10,} by model ({int(is_match[~labeled_mask].sum()):,} matches '
      f'at threshold {MATCH_THRESHOLD})')
print(f'   {int(is_match.sum()):>10,} matches in total '
      f'({is_match.mean():.2%} of all candidate pairs)')

3,008,719 pairs decided:
      853,135 by GTIN   (34,921 matches)
    2,155,584 by model (33,479 matches at threshold 0.8)
       68,400 matches in total (2.27% of all candidate pairs)


## 7. Grouping into items

In [197]:
# Creating a graph from the matching pairs.
edges = pairs[is_match]
graph = coo_matrix(
    (np.ones(len(edges), dtype=bool), (edges[:, 0], edges[:, 1])),
    shape=(len(df), len(df)))

# Finding connected components in the graph to group SKUs into items.
n_items, item_labels = connected_components(graph, directed=False)

# Counting the sizes of each connected component (item) and printing summary statistics.
sizes = np.bincount(item_labels)
print(f'{len(df):,} SKUs -> {n_items:,} items')
print(f'singletons {int((sizes == 1).sum()):,} ({(sizes == 1).mean():.1%}) | '
      f'largest item {sizes.max()} SKUs | mean size {sizes.mean():.2f}')

71,623 SKUs -> 41,618 items
singletons 28,990 (69.7%) | largest item 128 SKUs | mean size 1.72


### Writing the submission

In [198]:
# Writing the submission: one row per SKU, mapping it to the item it belongs to.
submission = pd.DataFrame({'sku_id': df['sku_id'], 'item_id': item_labels})

template = pd.read_csv('../Data/test_pred_template.csv')
assert len(submission) == len(template), 'row count differs from the template'
assert set(submission['sku_id']) == set(template['sku_id']), 'sku_id set differs'
assert submission['item_id'].notna().all(), 'every SKU needs an item_id'

submission.to_csv('../Data/predictions.csv', index=False)
print(f'wrote predictions.csv: {len(submission):,} rows, '
      f'{submission["item_id"].nunique():,} distinct items')
submission.head()

wrote predictions.csv: 71,623 rows, 41,618 distinct items


,sku_id,item_id
0,6916,0
1,12782,1
2,16627,2
3,19126,3
4,27474,4


## 8. Results

* Attempt 1: Match rate = 49.7302%
* Attempt 2: Match rate = 32.7947%
* Attempt 3: Match rate = 52.1302%